# NumCompute Quickstart Demo

This notebook demonstrates the main features of the NumCompute package using real CSV datasets.

We will cover:

1. Reading CSV files using `io.py`
2. Handling missing values
3. Preprocessing numerical and categorical data
4. Sorting, searching, top-k selection, quickselect, binary search, ranking and percentiles
5. Computing statistics such as mean, variance, histogram and quantiles
6. Estimating gradients and Jacobians using finite differences
7. Benchmarking vectorised NumPy implementations against Python loop implementations

For the demo, we use the Titanic dataset for CSV reading, missing value handling, preprocessing, sorting, ranking and statistics. We also use the Mall Customers dataset to demonstrate a full preprocessing pipeline.

## 1. Setup Project 

The notebook is stored inside the `demo/` folder, while the package source code is inside the project root folder.



In [ ]:
cd ..  #This should be run only once!

c:\Users\lincoln.basnet\Downloads\Cognitive_Coders\NumCompute


In [2]:
!pip install -e .

Obtaining file:///c:/Users/lincoln.basnet/Downloads/Cognitive_Coders/NumCompute
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for numcompute (pyproject.toml): started
  Building editable for numcompute (pyproject.toml): finished with status 'done'
  Created wheel for numcompute: filename=numcompute-0.1.0-0.editable-py3-none-any.whl size=2844 sha256=93ba117cc652872cc0e04de5e8f155704f4f00a7462626ebd920e3c58a6ea5f5
  Stored in directory: C:\Users\lincoln.basnet\AppData\Local\Temp\pip-ephem-wheel-cache-qwuj02v4\wheels\b3\58\b3\d

## 2. Import NumCompute Modules

In this section, we import the custom modules from the NumCompute package.

The demo uses:

- `io.py` for reading CSV files
- `preprocessing.py` for imputation, scaling and one-hot encoding
- `sort_search.py` for sorting, top-k, quickselect and binary search
- `rank.py` for ranking and percentile calculations
- `stats.py` for summary statistics
- `optim.py` for finite-difference gradients and Jacobians
- `pipeline.py` for combining preprocessing steps
- `benchmarking.py` for performance comparison

In [3]:
import numpy as np
from numcompute.io import CSVReader
from numcompute.preprocessing import StandardScaler, Imputer, OneHotEncoder
from numcompute.sort_search import sort, argsort, topk, quickselect, binary_search
from numcompute.sort_search import percentile as ss_percentile
from numcompute.rank import rank, percentile as rank_percentile
from numcompute.stats import mean, variance, histogram, quantile, describe, WelfordStatistics
from numcompute.optim import grad, jacobian

rank.py loaded ✓
metrics.py loaded ✓


In [4]:
from numcompute.sort_search import multikey_sort
from numcompute.pipeline import Pipeline
from numcompute.benchmarking import run_all_benchmarks

## 3. Reading a CSV File with `CSVReader`

Here we load the Titanic dataset using the custom `CSVReader` class from `io.py`.

This demonstrates that the package can:

- Read CSV files
- Handle a header row
- Detect missing values
- Convert numeric columns to NumPy arrays where possible

The Titanic dataset is useful here because it contains both numerical and categorical columns, as well as missing values.

In [5]:
# Load the Titanic dataset using the custom CSVReader
reader = CSVReader("./demo/titanic_dataset.csv", delimiter=",", has_header=True)
data = reader.read()

print("Dataset Shape:", data.shape)
print("\nFirst 3 Rows:")
print(data[:3])
print("\nHeader:", reader.header)

# Check which numeric columns contain missing values
print("\nMissing values per column (NaN detection via io.py):")
for i, col in enumerate(reader.header):
    try:
        col_data = data[:, i].astype(float)
        nan_count = np.sum(np.isnan(col_data))
    except (ValueError, TypeError):
        nan_count = 0
    if nan_count > 0:
        print(f"  [{i}] {col}: {nan_count} missing")

Dataset Shape: (418, 12)

First 3 Rows:
[['892' '0' '3' 'Kelly, Mr. James' 'male' '34.5' '0' '0' '330911'
  '7.8292' nan 'Q']
 ['893' '1' '3' 'Wilkes, Mrs. James (Ellen Needs)' 'female' '47' '1' '0'
  '363272' '7' nan 'S']
 ['894' '0' '2' 'Myles, Mr. Thomas Francis' 'male' '62' '0' '0' '240276'
  '9.6875' nan 'Q']]

Header: ['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked']

Missing values per column (NaN detection via io.py):
  [5] Age: 86 missing
  [9] Fare: 1 missing


## 4. Handling Missing Values with `Imputer`

The Titanic dataset contains missing values in numerical columns such as `Age` and sometimes `Fare`.

Before scaling or using these values in calculations, we replace missing values with the column mean using the custom `Imputer` class.

The correct preprocessing order is:

1. Extract numerical columns
2. Impute missing values
3. Apply scaling

In [6]:
# Extract numerical columns from Titanic:
# Age is column 5 and Fare is column 9 in this dataset.
age  = data[:, 5].astype(float)   
fare = data[:, 9].astype(float)

# Combine selected numerical columns into a feature matrix.
X_numeric = np.column_stack([age, fare])

print("Before imputation:")
print(f"  Age  NaNs: {np.sum(np.isnan(X_numeric[:, 0]))}")
print(f"  Fare NaNs: {np.sum(np.isnan(X_numeric[:, 1]))}")

# Replace missing values using the mean of each column.
imputer = Imputer(strategy="mean")
X_imputed = imputer.fit_transform(X_numeric)

print("\nAfter mean imputation:")
print(f"  Age  NaNs: {np.sum(np.isnan(X_imputed[:, 0]))}")
print(f"  Fare NaNs: {np.sum(np.isnan(X_imputed[:, 1]))}")
print(f"  Fill values used → Age: {imputer.statistics_[0]:.4f}, Fare: {imputer.statistics_[1]:.4f}")

Before imputation:
  Age  NaNs: 86
  Fare NaNs: 1

After mean imputation:
  Age  NaNs: 0
  Fare NaNs: 0
  Fill values used → Age: 30.2726, Fare: 35.6272


## 5. Standardising Numerical Features with `StandardScaler`

After missing values are filled, we standardise the numerical columns using `StandardScaler`.

Standardisation converts each numerical column so that it has approximately:

- Mean = 0
- Standard deviation = 1

This is commonly used before machine learning models because it puts numerical features on a comparable scale.

In [7]:
scaler = StandardScaler(with_mean=True, with_std=True)
X_scaled = scaler.fit_transform(X_imputed)

print("Scaled (Age, Fare) — first 5 rows:")
print(X_scaled[:5].round(4))
print("\nColumn means (should be ~0):", X_scaled.mean(axis=0).round(6))
print("Column stds  (should be ~1):", X_scaled.std(axis=0).round(6))

Scaled (Age, Fare) — first 5 rows:
[[ 0.335  -0.4984]
 [ 1.3255 -0.5133]
 [ 2.5142 -0.4651]
 [-0.2593 -0.4835]
 [-0.6555 -0.4185]]

Column means (should be ~0): [-0.  0.]
Column stds  (should be ~1): [1. 1.]


## 6. Encoding Categorical Features with `OneHotEncoder`

The Titanic dataset also contains categorical columns.

Here we encode:

- `Sex`
- `Embarked`

One-hot encoding converts text categories into numerical columns so they can be used in numerical pipelines or machine learning models.

In [8]:
# Select categorical columns:
# Sex is column 4.
# Embarked is column 11.
X_cat = data[:, [4, 11]].astype(str)

print("Unique values — Sex:      ", np.unique(X_cat[:, 0]))
print("Unique values — Embarked: ", np.unique(X_cat[:, 1]))

encoder = OneHotEncoder()
X_encoded = encoder.fit_transform(X_cat)

print("\nEncoded shape:", X_encoded.shape)   # (418, 5)
print("Categories learned:")
for i, cats in enumerate(encoder.categories_):
    print(f"  Column {i}: {cats}")
print("\nFirst 5 encoded rows:")
print(X_encoded[:5])

Unique values — Sex:       ['female' 'male']
Unique values — Embarked:  ['C' 'Q' 'S']

Encoded shape: (418, 5)
Categories learned:
  Column 0: ['female' 'male']
  Column 1: ['C' 'Q' 'S']

First 5 encoded rows:
[[0. 1. 0. 1. 0.]
 [1. 0. 0. 0. 1.]
 [0. 1. 0. 1. 0.]
 [0. 1. 0. 0. 1.]
 [1. 0. 0. 0. 1.]]


## 7. Sorting, Searching, Top-K and Quickselect

This section demonstrates functions from `sort_search.py`.

We use the Titanic `Fare` column as a numerical example.

The functions demonstrated are:

- `sort`
- `argsort`
- `topk`
- `quickselect`
- `binary_search`
- percentile calculation from `sort_search.py`

In [9]:
# Remove missing values before sorting and searching.
fare_clean = fare[~np.isnan(fare)]

# Sort fare values in ascending order.
sorted_fare = sort(fare_clean)
print("Sorted Fare (first 10):", sorted_fare[:10])

# Argsort returns the indices that would sort the original array.
indices = argsort(fare_clean)
print("Argsort indices (first 10):", indices[:10])

# Get the top 5 highest fare values
top_vals, top_idx = topk(fare_clean, k=5, largest=True)
print("\nTop-5 highest fares:", top_vals)
print("At indices:          ", top_idx)

# Quickselect finds the kth smallest value without fully sorting the array.
kth = quickselect(fare_clean, k=9)   # 10th smallest (0-based)
print("\n10th smallest fare (quickselect):", kth)

# Binary search requires sorted input.
idx, exists = binary_search(sorted_fare, 15.5)
print(f"\nBinary search for 15.5 → index: {idx}, exact match: {exists}")

# Percentile calculation from sort_search.py
p90 = ss_percentile(fare_clean, 90)
print(f"90th percentile of Fare (sort_search): {p90:.4f}")

Sorted Fare (first 10): [0.     0.     3.1708 6.4375 6.4375 6.4958 6.95   7.     7.     7.05  ]
Argsort indices (first 10): [265 371  21 116 133 231 290   1 162 210]

Top-5 highest fares: [512.3292 263.     263.     262.375  262.375 ]
At indices:           [342  69  53 374  64]

10th smallest fare (quickselect): 7.05

Binary search for 15.5 → index: 223, exact match: True
90th percentile of Fare (sort_search): 79.2000


## 8. Multikey Sorting

In this section, we demonstrate multikey sorting.

We sort Titanic passengers using two numerical columns:

1. `Pclass`
2. `Age`

This is useful when we want to sort rows by one column first, and then use another column as a tie-breaker.

In [10]:

# Sort passengers by Pclass first and Age second.
numeric_data = np.column_stack([
    data[:, 2].astype(float),   
    X_imputed[:, 0]                          
])
sorted_data = multikey_sort(numeric_data, columns=[0, 1], ascending=True)
print("\nMultikey sort — by Pclass then Age (first 5 rows):")
print(sorted_data[:5])


Multikey sort — by Pclass then Age (first 5 rows):
[[ 1.  6.]
 [ 1. 13.]
 [ 1. 17.]
 [ 1. 18.]
 [ 1. 18.]]


## 9. Ranking and Tie Handling

This section demonstrates ranking methods from `rank.py`.

Tie handling is important because repeated values can be ranked in different ways.

We compare:

- `average`: tied values receive the average of their rank positions
- `dense`: tied values share the same rank, and the next distinct value gets the next rank
- `ordinal`: tied values are ranked based on their order of appearance

We also calculate percentiles using the `rank.py` percentile function.

In [11]:
sample_scores = np.array([100, 90, 90, 80, 75, 75, 60], dtype=float)
print("Scores:", sample_scores)
print()
print("Average ranks (ties share mean rank):", rank(sample_scores, method="average"))
print("Dense ranks   (ties share next rank):", rank(sample_scores, method="dense"))
print("Ordinal ranks (ties broken by order):", rank(sample_scores, method="ordinal"))

print("\nPercentiles of Fare (rank.py, with ignore_nan):")
print(f"  25th: {rank_percentile(fare, 25, ignore_nan=True):.4f}")
print(f"  50th: {rank_percentile(fare, 50, ignore_nan=True):.4f}")
print(f"  75th: {rank_percentile(fare, 75, ignore_nan=True):.4f}")
print(f"  90th: {rank_percentile(fare, 90, ignore_nan=True):.4f}")

Scores: [100.  90.  90.  80.  75.  75.  60.]

Average ranks (ties share mean rank): [7.  5.5 5.5 4.  2.5 2.5 1. ]
Dense ranks   (ties share next rank): [5. 4. 4. 3. 2. 2. 1.]
Ordinal ranks (ties broken by order): [7. 5. 6. 4. 2. 3. 1.]

Percentiles of Fare (rank.py, with ignore_nan):
  25th: 7.8958
  50th: 14.4542
  75th: 31.5000
  90th: 79.2000


## 10. Statistical Summary Functions

This section demonstrates functions from `stats.py`.

Using the Titanic `Fare` column, we calculate:

- Mean
- Variance
- Quantiles
- Histogram bins
- Streaming statistics using Welford’s algorithm
- A full descriptive summary using `describe()`

The `ignore_nan=True` option allows the functions to safely handle missing values.

In [12]:
# Basic summary statistics
print("Mean of Fare:    ", mean(fare, ignore_nan=True))
print("Variance of Fare:", variance(fare, ddof=1, ignore_nan=True))

# Quantiles show the distribution at selected cut points
q25 = quantile(fare, 0.25, ignore_nan=True)
q50 = quantile(fare, 0.50, ignore_nan=True)
q75 = quantile(fare, 0.75, ignore_nan=True)
print(f"\nQuantiles → Q1: {q25:.4f}, Median: {q50:.4f}, Q3: {q75:.4f}")

# Histogram groups fare values into bins
counts, edges = histogram(fare_clean, bins=8)
print("\nHistogram (8 bins):")
for i in range(len(counts)):
    print(f"  [{edges[i]:.2f} – {edges[i+1]:.2f}]: {int(counts[i])} passengers")

# Welford's algorithm calculates streaming statistics
ws = WelfordStatistics()
ws.update_batch(fare_clean)          
print(f"\nWelford streaming stats over {ws.n} values:")
print(f"  Mean:   {float(ws.mean()):.4f}")
print(f"  Std:    {float(ws.std(ddof=1)):.4f}")
print(f"  Var:    {float(ws.variance(ddof=1)):.4f}")

# describe() returns a dictionary of multiple summary statistics
print("\nFull describe():")
for k, v in describe(fare).items():
    print(f"  {k}: {v}")

Mean of Fare:     35.627188489208635
Variance of Fare: 3125.657074319579

Quantiles → Q1: 7.8958, Median: 14.4542, Q3: 31.5000

Histogram (8 bins):
  [0.00 – 64.04]: 359 passengers
  [64.04 – 128.08]: 29 passengers
  [128.08 – 192.12]: 11 passengers
  [192.12 – 256.16]: 10 passengers
  [256.16 – 320.21]: 7 passengers
  [320.21 – 384.25]: 0 passengers
  [384.25 – 448.29]: 0 passengers
  [448.29 – 512.33]: 1 passengers

Welford streaming stats over 417 values:
  Mean:   35.6272
  Std:    55.9076
  Var:    3125.6571

Full describe():
  n: 417
  nan_count: 1
  mean: 35.627188489208635
  std: 55.907576179973844
  variance: 3125.657074319579
  min: 0.0
  max: 512.3292
  q25: 7.8958
  median: 14.4542
  q75: 31.5
  iqr: 23.6042
  skew: 3.6739366758439074
  kurtosis: 17.693074169762273


## 11. Numerical Gradients and Jacobians with `optim.py`

This section demonstrates finite-difference numerical differentiation.

We calculate:

1. The gradient of a scalar-valued function
2. The Jacobian matrix of a vector-valued function

We compare the numerical results with analytical results to confirm that the implementation is accurate.

In [13]:
def scalar_fn(x):
    """
    Evaluate a scalar-valued function.

    Parameters
    ----------
    x : np.ndarray
        A one-dimensional input array with two values: x[0] and x[1].

    Returns
    -------
    float
        The scalar function value f(x, y) = x^2 + 3y^2 + sin(x).
    """
    return x[0]**2 + 3*x[1]**2 + np.sin(x[0])

x0 = np.array([2.0, 3.0])

# Estimate the gradient using central and forward finite differences.
g_central = grad(scalar_fn, x0, h=1e-5, method="central")
g_forward  = grad(scalar_fn, x0, h=1e-5, method="forward")

# Analytical gradient:
# df/dx = 2x + cos(x)
# df/dy = 6y
g_analytic = np.array([2*x0[0] + np.cos(x0[0]), 6*x0[1]])

print("Scalar function: f(x,y) = x² + 3y² + sin(x)  at", x0)
print(f"Central gradient: {g_central}")
print(f"Forward gradient: {g_forward}")
print(f"Analytic:         {g_analytic}")
print(f"Central error:    {np.abs(g_central - g_analytic)}")

def vector_fn(x):
    """
    Evaluate a vector-valued function.

    Parameters
    ----------
    x : np.ndarray
        A one-dimensional input array with two values: x[0] and x[1].

    Returns
    -------
    np.ndarray
        A vector output F(x, y) = [x^2 + y, sin(x) + y^2].
    """
    return np.array([x[0]**2 + x[1], np.sin(x[0]) + x[1]**2])

# Estimate the Jacobian matrix using central finite differences.
J_est = jacobian(vector_fn, x0, h=1e-5, method="central")
J_exact = np.array([[2*x0[0], 1], [np.cos(x0[0]), 2*x0[1]]])

print("\nVector function: F(x,y) = [x²+y, sin(x)+y²]  at", x0)
print("  Estimated Jacobian:\n", J_est)
print("  Analytic Jacobian:\n", J_exact)
print("  Max error:", np.abs(J_est - J_exact).max())

Scalar function: f(x,y) = x² + 3y² + sin(x)  at [2. 3.]
Central gradient: [ 3.58385316 18.        ]
Forward gradient: [ 3.58385862 18.00003   ]
Analytic:         [ 3.58385316 18.        ]
Central error:    [1.16159526e-10 2.91038305e-11]

Vector function: F(x,y) = [x²+y, sin(x)+y²]  at [2. 3.]
  Estimated Jacobian:
 [[ 4.          1.        ]
 [-0.41614684  6.        ]]
  Analytic Jacobian:
 [[ 4.          1.        ]
 [-0.41614684  6.        ]]
  Max error: 3.930633596382904e-11


## 12. Full Preprocessing Pipeline with Mall Customers Dataset

This section demonstrates how preprocessing steps can be combined using the custom `Pipeline` class.

We use the Mall Customers dataset because it contains:

- Numerical features: `Age`, `Annual Income`, `Spending Score`
- Categorical feature: `Gender`

The numerical pipeline applies:

1. Mean imputation
2. Standard scaling

The categorical pipeline applies:

1. One-hot encoding

Finally, the numerical and categorical outputs are combined into one processed feature matrix.

In [14]:
reader2 = CSVReader("./demo/Mall_Customers.csv", delimiter=",", has_header=True)
data2 = reader2.read()

print("Mall Customers header:", reader2.header)
print("Shape:", data2.shape)
print("\nFirst 5 rows (raw):")
print(data2[:5])

# Numerical features: Age[2], Income[3], Score[4]
X_mall = np.column_stack([
    data2[:, 2].astype(float),
    data2[:, 3].astype(float),
    data2[:, 4].astype(float),
])

print("\nMissing values before pipeline:")
print(f"  Age NaNs:    {np.sum(np.isnan(X_mall[:, 0]))}")
print(f"  Income NaNs: {np.sum(np.isnan(X_mall[:, 1]))}")
print(f"  Score NaNs:  {np.sum(np.isnan(X_mall[:, 2]))}")

# Numerical pipeline: impute missing values first, then scale.
pipe_num = Pipeline([
    ("imputer", Imputer(strategy="mean")),
    ("scaler",  StandardScaler(with_mean=True, with_std=True)),
])
X_num_out = pipe_num.fit_transform(X_mall)

print("\nNumerical Pipeline:", pipe_num)
print("  Output shape:", X_num_out.shape)
print("  NaNs remaining:", np.sum(np.isnan(X_num_out)))
print("  Column means (should be ~0):", X_num_out.mean(axis=0).round(6))
print("  Column stds  (should be ~1):", X_num_out.std(axis=0).round(6))

X_genre = data2[:, [1]].astype(str)
X_genre[X_genre == 'nan'] = 'Unknown'

# Categorical pipeline: one-hot encode gender.
pipe_cat = Pipeline([
    ("encoder", OneHotEncoder()),
])
X_cat_out = pipe_cat.fit_transform(X_genre)

print("\nCategorical Pipeline:", pipe_cat)
print("  Output shape:", X_cat_out.shape)
print("  Categories learned:", pipe_cat.steps[0][1].categories_)

X_final = np.hstack([X_num_out, X_cat_out])
print("\nFinal combined feature matrix shape:", X_final.shape)
print("First 5 rows:")
print(X_final[:5].round(4))

Mall Customers header: ['CustomerID', 'Genre', 'Age', 'Annual Income (k$)', 'Spending Score (1-100)']
Shape: (200, 5)

First 5 rows (raw):
[[nan 'Male' '19' '15' '39']
 ['0002' nan '21' nan '81']
 ['0003' 'Female' nan '16' '6']
 ['0004' nan '23' '16' nan]
 ['0005' 'Female' '31' '17' '40']]

Missing values before pipeline:
  Age NaNs:    1
  Income NaNs: 1
  Score NaNs:  1

Numerical Pipeline: Pipeline(steps=['imputer', 'scaler'])
  Output shape: (200, 3)
  NaNs remaining: 0
  Column means (should be ~0): [0. 0. 0.]
  Column stds  (should be ~1): [1. 1. 1.]

Categorical Pipeline: Pipeline(steps=['encoder'])
  Output shape: (200, 3)
  Categories learned: [array(['Female', 'Male', 'Unknow'], dtype='<U6')]

Final combined feature matrix shape: (200, 6)
First 5 rows:
[[-1.438  -1.7612 -0.4307  0.      1.      0.    ]
 [-1.2938  0.      1.2042  0.      0.      1.    ]
 [ 0.     -1.7227 -1.7154  1.      0.      0.    ]
 [-1.1496 -1.7227  0.      0.      0.      1.    ]
 [-0.5728 -1.6842 -0.39

## 13. Benchmarking Vectorised Implementations Against Python Loops

This section benchmarks NumCompute’s vectorised implementations against equivalent Python loop implementations.

The benchmark compares execution time across different input sizes.

This helps demonstrate why vectorised NumPy operations are preferred for numerical computing tasks.

In [15]:
# ── 10. BENCHMARKING ──────────────────────────────────────────────────────────
results = run_all_benchmarks(sizes=[1_000, 10_000, 100_000, 500_000], n_runs=20, seed=42)

Environment
  Python : 3.14.3
  NumPy  : 2.4.4
  OS     : Windows 11

------------------------------------------------------------------
 NumCompute — vectorised vs Python loop benchmarks
------------------------------------------------------------------

── mean ──
Size         numcompute (s)     loop (s)           Speedup   
-------------------------------------------------------------
1000         0.000025           0.000045           1.78      x
10000        0.000007           0.000483           70.48     x
100000       0.000027           0.007578           278.74    x
500000       0.000198           0.032437           163.90    x

── std ──
Size         numcompute (s)     loop (s)           Speedup   
-------------------------------------------------------------
1000         0.000026           0.000292           11.31     x
10000        0.000036           0.002869           79.04     x
100000       0.000257           0.029997           116.52    x
500000       0.005167           0